In [15]:
from read_model_runs import read_model_runs

# Ler dados completos
question_long_df = \
    read_model_runs('../../data/processed/model-runs')



In [16]:
from read_model_runs import filter_complete_questions

models = ['gemma-3-27b-it', 'gemma-3-12b-it', 'gemma-3-4b-it']
firacs = ['FILA_', 'FIR__', 'FI___', 'FIL__', '_____', 'unstructured']
question_long_df, question_wide_df, model_order, firac_order = filter_complete_questions(question_long_df, models, firacs)

print('shape:', question_long_df.shape)
print('# unique questions:', question_long_df['question_id'].nunique())
print("model order:", model_order)
print("firac order:", firac_order)


shape: (40788, 32)
# unique questions: 2266
model order: ['gemma-3-4b-it', 'gemma-3-12b-it', 'gemma-3-27b-it']
firac order: ['_____', 'unstructured', 'FIL__', 'FI___', 'FIR__', 'FILA_']


In [17]:
question_wide_df.head()

,question_id,model_name,_____,unstructured,FIL__,FI___,FIR__,FILA_
0,oab-1.pdf-002,gemma-3-12b-it,False,True,True,True,True,True
1,oab-1.pdf-002,gemma-3-27b-it,False,True,True,True,True,True
2,oab-1.pdf-002,gemma-3-4b-it,True,True,True,True,True,True
3,oab-1.pdf-003,gemma-3-12b-it,False,False,False,False,True,True
4,oab-1.pdf-003,gemma-3-27b-it,False,False,True,True,True,True


In [18]:
question_wide_df.columns

Index(['question_id', 'model_name', '_____', 'unstructured', 'FIL__', 'FI___',
       'FIR__', 'FILA_'],
      dtype='object')

In [19]:
import pandas as pd

pca_df = pd.read_csv("../../data/processed/pca.csv")

pca_df.head()

,Unnamed: 0,materia,tema,oab_test_id,question_id,PC1,PC2,PC3,PC4,PC5,...,PC9,PC10,PC11,PC12,PC13,PC14,PC15,PC16,PC17,PC18
0,0,ÉTICA PROFISSIONAL,DIREITOS DO ADVOGADO,XV,oab-1.pdf-002,1.961544,0.240658,1.446796,0.133590,0.040381,...,0.085222,0.393587,0.578057,-0.442549,-1.048116,-0.749840,1.370845,-0.523608,-0.561300,0.094604
1,1,ÉTICA PROFISSIONAL,ATIVIDADE DE ADVOCACIA,XV,oab-1.pdf-003,-2.204211,1.833674,-0.357334,0.652696,-1.322534,...,0.284110,0.425735,1.358740,0.863399,0.572882,-0.531869,0.369861,-0.031478,-0.133353,0.090464
2,2,ÉTICA PROFISSIONAL,SOCIEDADE DE ADVOGADOS,XV,oab-1.pdf-004,1.124033,0.475079,-1.302193,0.032035,-0.516841,...,0.845768,1.082327,0.841169,-1.264569,-0.655066,0.295634,0.481186,-1.035932,0.408953,-0.094785
3,3,ÉTICA PROFISSIONAL,ATIVIDADE DE ADVOCACIA,XIV,oab-1.pdf-005,0.122749,-6.622188,-0.257877,0.142605,0.794772,...,2.569862,-3.389400,0.483281,1.413317,-1.753756,0.114902,-0.371465,0.349417,-0.355340,0.185112
4,4,ÉTICA PROFISSIONAL,ÉTICA DO ADVOGADO,XIV,oab-1.pdf-006,2.122710,-0.097887,-0.831125,-0.179571,0.074136,...,0.842472,1.148365,0.356519,0.301641,-0.045479,-0.061809,-0.144858,-0.051201,-0.031518,-0.081893


In [20]:
import pandas as pd
import numpy as np

df = pca_df.copy()

required_cols = {"PC1", "PC2", "materia", "tema", "question_id"}
assert required_cols.issubset(df.columns)


In [21]:
import pandas as pd
import json

def show_question(question_long_df, question_id):
    exam_df = pd.read_csv(f"../../data/processed/oab_with_firac_portuguese_shuffle.csv")
    exam_df = exam_df[exam_df.question_id == question_id]

    filtered_df = question_long_df[question_long_df.question_id == question_id]
    filtered_df = filtered_df[filtered_df.firac == '_____']

    question = exam_df.iloc[0]

    print(f"Enunciado: {question['enunciado']}\n")
    print(f"A) {question['A']}")
    print(f"B) {question['B']}")
    print(f"C) {question['C']}")
    print(f"D) {question['D']}")

    # Imprime todos os Facts com o modelo
    print("\n=== Facts ===")
    print(f"Gabarito: {question['Facts']}")
    print("*****")
    for index, row in filtered_df.iterrows():
        print(f"Modelo: {row['model_name']}, {row['Facts']}")

    # Imprime todos os Issues com o modelo
    print("\n=== Issue ===")
    print(f"Gabarito: {question['Issue']}")
    print("*****")
    for index, row in filtered_df.iterrows():
        print(f"Modelo: {row['model_name']}, {row['Issue']}")

    # Imprime todas as Rules com o modelo
    print("\n=== Rule ===\nGabarito:\n")
    print("".join(f"{k} → {item}" for k, v in json.loads(question['Rule']).items() if v for item in v))
    print("*****")
    for index, row in filtered_df.iterrows():
        print(f"Modelo: {row['model_name']} -- {row['Rule']}")

    # Imprime todas as Applications com o modelo
    print("\n=== Application ===")
    print(f"Gabarito: {question['Application']}")
    print("*****")
    for index, row in filtered_df.iterrows():
        print(f"Modelo: {row['model_name']} -- {row['Application']}")

    # Imprime todas as Conclusions com o modelo
    print("\n=== Conclusion ===")
    print(f"Gabarito: {question['Conclusion']}")
    print("*****")
    for index, row in filtered_df.iterrows():
        print(f"Modelo: {row['model_name']} -- {row['Conclusion']}")


    print("\n=== Resposta ===")
    print(f"Gabarito: {question['correct_option']}")
    print("*****")
    for index, row in filtered_df.iterrows():
        print(f"Modelo: {row['model_name']} -- {row['chosen_option']}, {row[row['chosen_option']]}")


In [22]:
show_question(question_long_df, "oab-167.pdf-150")

Enunciado: Antônio, vendedor, celebrou contrato de compra e venda com Joaquim, comprador, no
dia 1º de setembro de 2016, cujo objeto era um carro da marca X no valor de R$
20.000,00, sendo o pagamento efetuado à vista na data de assinatura do contrato.
Ficou estabelecido ainda que a entrega do bem seria feita 30 dias depois, em 1º de
outubro de 2016, na cidade do Rio de Janeiro, domicílio do vendedor. Contudo, no dia
25 de setembro, uma chuva torrencial inundou diversos bairros da cidade e o carro foi
destruído pela enchente, com perda total. Considerando a descrição dos fatos,
Joaquim

A) terá direito à devolução de 50% do valor, tendo em vista que Antônio, vendedor,
teve culpa.
B) terá direito à devolução de 100% do valor, pois ainda não havia ocorrido a
tradição no momento do perecimento do bem.
C) terá direito à devolução de 50% do valor, tendo em vista que Antônio, vendedor,
não teve culpa.
D) não faz jus à devolução do pagamento de R$ 20.000,00.

=== Facts ===
Gabarito: [
  "Antô

In [23]:
import numpy as np

def assign_quadrant(sub_df):
    pc1_med = 0 # sub_df["PC1"].median()
    pc2_med = 0 # sub_df["PC2"].median()

    conditions = [
        (sub_df["PC1"] >= pc1_med) & (sub_df["PC2"] >= pc2_med),
        (sub_df["PC1"] >= pc1_med) & (sub_df["PC2"] <  pc2_med),
        (sub_df["PC1"] <  pc1_med) & (sub_df["PC2"] >= pc2_med),
        (sub_df["PC1"] <  pc1_med) & (sub_df["PC2"] <  pc2_med),
    ]

    # PC1 alto = fácil | PC1 baixo = difícil
    labels = [
        "easy_processual",    # PC1 alto, PC2 alto
        "easy_material",      # PC1 alto, PC2 baixo
        "hard_processual",    # PC1 baixo, PC2 alto
        "hard_material",      # PC1 baixo, PC2 baixo
    ]

    sub_df = sub_df.copy()
    sub_df["quadrant"] = np.select(conditions, labels)
    return sub_df


df = (
    df
    .groupby("materia", group_keys=False)
    .apply(assign_quadrant)
)


C:\Users\pedro\AppData\Local\Temp\ipykernel_16064\1112562615.py:30: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(assign_quadrant)


In [ ]:
def sample_quadrant(sub_df, n=3):
    centroid = sub_df[["PC1", "PC2"]].mean()
    sub_df = sub_df.copy()

    sub_df["dist_to_centroid"] = np.sqrt(
        (sub_df["PC1"] - centroid["PC1"])**2 +
        (sub_df["PC2"] - centroid["PC2"])**2
    )

    return sub_df.nlargest(n, "dist_to_centroid")


In [25]:
N_PER_QUADRANT = 1  # ajuste se quiser 3

samples = (
    df
    .groupby(["materia", "quadrant"], group_keys=False)
    .apply(sample_quadrant, n=N_PER_QUADRANT)
    .reset_index(drop=True)
)


C:\Users\pedro\AppData\Local\Temp\ipykernel_16064\2005124743.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(sample_quadrant, n=N_PER_QUADRANT)


In [26]:
final_sample_df = samples[
    [
        "materia",
        "quadrant",
        "question_id",
        "tema",
        "PC1",
        "PC2",
        "dist_to_centroid",
    ]
].sort_values(["materia", "quadrant", "dist_to_centroid"])

print(final_sample_df.shape)

final_sample_df[final_sample_df.materia == 'FILOSOFIA']

(78, 7)


,materia,quadrant,question_id,tema,PC1,PC2,dist_to_centroid
59,FILOSOFIA,easy_material,oab-40.pdf-004,JUSFILÓSOFOS MODERNOS,3.098099,-0.335001,0.195606
60,FILOSOFIA,easy_processual,oab-42.pdf-030,PRINCIPAIS CORRENTES FILOSÓFICAS,1.231621,0.421759,0.384678
61,FILOSOFIA,hard_material,oab-40.pdf-011,ÉTICA E MORAL,-1.312918,-1.171212,0.000000


In [27]:
show_question(question_long_df, "oab-222.pdf-040")

Enunciado: “Toda pessoa que se acha no exercício dos seus direitos tem capacidade para estar em
juízo”, estabelece o Código de Processo Civil, e os incapazes serão assistidos ou
representados por seus pais, tutores ou curadores. A respeito do tema estão corretas
as afirmativas a seguir, à exceção de uma. Assinale-a.

A) O juiz dará curador especial ao incapaz, ainda que tenha representante legal,
quando houver colisão de interesses entre este e o representado.
B) Ao curador especial não se aplica o ônus da impugnação especificada dos fatos
articulados pelo autor.
C) O juiz dará curador especial ao réu revel citado por edital, mas não àquele citado
com hora certa.
D) O curador especial, nomeado em caso de executado citado com hora certa revel,
tem legitimidade para opor embargos à execução.

=== Facts ===
Gabarito: [
  "Pessoas no exercício de seus direitos possuem capacidade para estar em juízo.",
  "Indivíduos incapazes devem ser assistidos ou representados por seus pais, tutores ou c